<a href="https://colab.research.google.com/github/tamanna155/WildFire_Detection/blob/master/RESNET_50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torchvision import models

def get_resnet50_wildfire_model(num_classes=3):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

    # Replace the classifier head
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amerzishminha/forest-fire-smoke-and-non-fire-image-dataset")

print("Path to dataset files:", path)

100%|██████████| 6.43G/6.43G [04:58<00:00, 23.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/amerzishminha/forest-fire-smoke-and-non-fire-image-dataset/versions/3


In [3]:
!cp -r /root/.cache/kagglehub/datasets/amerzishminha/forest-fire-smoke-and-non-fire-image-dataset/versions/3/FOREST_FIRE_SMOKE_AND_NON_FIRE_DATASET /content/

In [4]:
import os, shutil
import random
from pathlib import Path

def split_dataset(train_dir, val_dir, val_ratio=0.2):
    classes = os.listdir(train_dir)

    for cls in classes:
        cls_train = Path(train_dir) / cls
        cls_val = Path(val_dir) / cls
        cls_val.mkdir(parents=True, exist_ok=True)

        images = list(cls_train.glob("*"))
        val_count = int(len(images) * val_ratio)

        val_images = random.sample(images, val_count)

        for img_path in val_images:
            shutil.move(str(img_path), cls_val / img_path.name)

split_dataset("/content/FOREST_FIRE_SMOKE_AND_NON_FIRE_DATASET/train", "/content/FOREST_FIRE_SMOKE_AND_NON_FIRE_DATASET/val", val_ratio=0.2)

In [5]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch.optim as optim

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Dataset
train_dataset = ImageFolder(root='/content/FOREST_FIRE_SMOKE_AND_NON_FIRE_DATASET/train', transform=transform)
val_dataset = ImageFolder(root='/content/FOREST_FIRE_SMOKE_AND_NON_FIRE_DATASET/val', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [6]:
# Model
model = get_resnet50_wildfire_model(num_classes=3).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training loop
for epoch in range(10):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 125MB/s]


Streaming output truncated to the last 5000 lines.
Epoch 4, Loss: 14.8228
Epoch 4, Loss: 14.8374
Epoch 4, Loss: 14.8564
Epoch 4, Loss: 14.8575
Epoch 4, Loss: 14.9131
Epoch 4, Loss: 14.9557
Epoch 4, Loss: 14.9747
Epoch 4, Loss: 15.0527
Epoch 4, Loss: 15.0546
Epoch 4, Loss: 15.0579
Epoch 4, Loss: 15.0753
Epoch 4, Loss: 15.1182
Epoch 4, Loss: 15.1388
Epoch 4, Loss: 15.1563
Epoch 4, Loss: 15.1576
Epoch 4, Loss: 15.1778
Epoch 4, Loss: 15.1788
Epoch 4, Loss: 15.1981
Epoch 4, Loss: 15.2155
Epoch 4, Loss: 15.2977
Epoch 4, Loss: 15.2984
Epoch 4, Loss: 15.4455
Epoch 4, Loss: 15.4884
Epoch 4, Loss: 15.4956
Epoch 4, Loss: 15.5187
Epoch 4, Loss: 15.5646
Epoch 4, Loss: 15.5948
Epoch 4, Loss: 15.6040
Epoch 4, Loss: 15.6074
Epoch 4, Loss: 15.6257
Epoch 4, Loss: 15.6262
Epoch 4, Loss: 15.6304
Epoch 4, Loss: 15.6314
Epoch 4, Loss: 15.6675
Epoch 4, Loss: 15.6685
Epoch 4, Loss: 15.7265
Epoch 4, Loss: 15.7325
Epoch 4, Loss: 15.7354
Epoch 4, Loss: 15.7414
Epoch 4, Loss: 15.7430
Epoch 4, Loss: 15.7526
Epoch 

In [7]:
def evaluate(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print(f'Validation Accuracy: {100 * correct / total:.2f}%')

evaluate(model, val_loader)


Validation Accuracy: 98.36%


In [8]:
torch.save(model.state_dict(), 'resnet50_wildfire.pth')
